# Raw-concatenation MLP: four 128-unit hidden layers

Reuse the saved generations, BLEURT-20 scores, and 29 × 3,584 hidden states for all six training domains and held-out TriviaQA. No projections and no new Qwen generation. The classifier has **four hidden linear layers of width 128**, followed by one 1-unit output layer. It retains the residual-block pattern, LayerNorm, Mish, dropout 0.2, BCE, AdamW, 40 epochs, and all prior splits/evaluation. Attach the private dataset `nvthaai/sala-g14-6-source-triviaqa-version-6-artifacts` and select GPU T4 x2 before Save & Run All.

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!git checkout f19dad7
!pip install -q -r requirements-kaggle.txt

In [ ]:
from pathlib import Path
SOURCE_NAME = 'sala_qwen25_g14_6src_triviaqa_v1'
domains = ['tqa', 'nq_open', 'sciq', 'popqa', 'web_questions', 'hotpotqa', 'triviaqa']
artifact_dirs = [p for p in Path('/kaggle/input').rglob(SOURCE_NAME) if p.name == SOURCE_NAME and (p / 'generation').is_dir()]
if len(artifact_dirs) != 1:
    raise RuntimeError(f'Expected one saved artifact set; found {artifact_dirs}')
ARTIFACT_DIR = artifact_dirs[0]
required = [*(ARTIFACT_DIR / 'generation' / d / f for d in domains for f in ('rows.jsonl', 'features.npy', 'manifest.json')), *(ARTIFACT_DIR / 'bleurt' / d / f for d in domains for f in ('scores.jsonl', 'manifest.json'))]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise RuntimeError('Missing artifact files: ' + ', '.join(missing))
print('Reusing saved source and target generations:', ARTIFACT_DIR)

In [ ]:
import subprocess, sys
OUTPUT_DIR = Path('/kaggle/working/sala_g14_6src_raw_concat_4x128')
command = [sys.executable, '/kaggle/working/SALA/sala_raw_concat.py', '--input-artifact-dir', str(ARTIFACT_DIR), '--output-dir', str(OUTPUT_DIR), '--bleurt-threshold', '0.5', '--epochs-erm', '40', '--seed', '42', '--train-device', 'cuda:0', '--hidden-dim', '128', '--hidden-layers', '4']
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
import json, pandas as pd
results = pd.read_csv(OUTPUT_DIR / 'results.csv')
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
print('Rows:', len(results), 'MLP input:', summary['mlp_input_dimension'], 'hidden layers:', summary['mlp_hidden_layers'], 'width:', summary['mlp_hidden_dimension'])
print(json.dumps(summary['target_metrics'], indent=2))
display(results.head())